# K-Nearest Neighbors

**Companion lesson:** https://ml-viz.vercel.app/courses/knn-decision-trees/01-knn

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## KNN from scratch

Classify by majority vote of the k nearest neighbors. We first reproduce the
exact worked example from the lesson: training points A..E and the query (3, 4).

In [ ]:
class KNN:
    """K-Nearest Neighbors classifier (integer labels)."""

    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []
        for x in X:
            # Euclidean distance from x to every training point.
            dists = np.linalg.norm(self.X_train - x, axis=1)
            idx = np.argsort(dists)[:self.k]      # indices of k nearest
            votes = self.y_train[idx]
            preds.append(np.bincount(votes).argmax())  # majority vote
        return np.array(preds)


# ---- Reproduce the worked example from the lesson ----
# Training points A..E and the query point (3, 4).
# Labels: Red = 0, Blue = 1.
X_lesson = np.array([[1, 2], [2, 3], [6, 5], [7, 7], [8, 6]], dtype=float)
y_lesson = np.array([0, 0, 1, 1, 1])          # A,B Red ; C,D,E Blue
names = np.array(['A', 'B', 'C', 'D', 'E'])
label_name = {0: 'Red', 1: 'Blue'}

query = np.array([3.0, 4.0])
dists = np.linalg.norm(X_lesson - query, axis=1)

order = np.argsort(dists)
print('Distances from query (3, 4), nearest first:')
for rank, i in enumerate(order, start=1):
    print('  {}. {}  d={:.2f}  ({})'.format(rank, names[i], dists[i], label_name[y_lesson[i]]))

for k in (3, 5):
    pred = KNN(k=k).fit(X_lesson, y_lesson).predict(query[None, :])[0]
    print('k={}: predict {}'.format(k, label_name[pred]))

## Decision boundary on synthetic data

Now build a larger two-cluster dataset and draw the boundary KNN carves out.

In [ ]:
np.random.seed(42)
n = 60
X0 = np.random.randn(n // 2, 2) + [-1.5, 0]
X1 = np.random.randn(n // 2, 2) + [1.5, 0]
X = np.vstack([X0, X1])
y = np.array([0] * (n // 2) + [1] * (n // 2))

# Dense grid we will classify to colour the background.
xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-5, 5, 100))
grid = np.c_[xx.ravel(), yy.ravel()]

knn = KNN(k=5).fit(X, y)
Z = knn.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlBu')
ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=20, alpha=0.7, label='Class 0')
ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=20, alpha=0.7, label='Class 1')
ax.legend()
ax.set_title('KNN Decision Boundary (k=5)', color='white')
plt.tight_layout()
plt.show()

## Effect of k on the decision boundary

Small k bends around every point (low bias, high variance). Large k smooths the
boundary (high bias, low variance) until k=n always predicts the majority class.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, k in zip(axes, [1, 3, 7, 20]):
    knn = KNN(k=k).fit(X, y)
    Z = knn.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlBu')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=10, alpha=0.5)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=10, alpha=0.5)
    ax.set_title('k = {}'.format(k), color='white')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
plt.suptitle('KNN: Effect of k on Decision Boundary', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Distance metrics matter

Euclidean (L2), Manhattan (L1), and Minkowski (general p) change which neighbors
count as 'closest'. We reproduce the lesson's worked computation for the gaps
(3, 4, 0) between x = (1, 2, 3) and y = (4, 6, 3).

In [ ]:
def minkowski(x, y, p):
    return np.sum(np.abs(x - y) ** p) ** (1.0 / p)


x = np.array([1.0, 2.0, 3.0])
y = np.array([4.0, 6.0, 3.0])
gaps = np.abs(x - y)
print('per-coordinate gaps:', gaps)               # [3. 4. 0.]

print('Manhattan  (p=1):', minkowski(x, y, 1))    # 7.0
print('Euclidean  (p=2):', minkowski(x, y, 2))    # 5.0
print('Minkowski  (p=3): {:.2f}'.format(minkowski(x, y, 3)))  # ~4.50
print('Chebyshev (p->inf, = max gap):', gaps.max())           # 4.0

# Cross-check Euclidean / Manhattan against NumPy built-ins.
print('check L2:', np.linalg.norm(x - y))
print('check L1:', np.abs(x - y).sum())

## The curse of dimensionality

In high dimensions, distances concentrate: the nearest and farthest points
become almost equally far, so the ratio (max distance / min distance) collapses
toward 1 and 'nearest neighbor' loses meaning.

In [ ]:
rng = np.random.default_rng(0)
print('500 random points per dimension; ratio = farthest / nearest neighbor')
for d in [2, 10, 100, 1000]:
    pts = rng.random((500, d))
    dists = np.linalg.norm(pts - pts[0], axis=1)[1:]   # drop self-distance (0)
    ratio = dists.max() / dists.min()
    print('dim={:>4}: (max/min distance) = {:.2f}'.format(d, ratio))

## Key takeaways

- KNN is **lazy**: no training — it stores the data and votes among the $k$ nearest points at query time.
- Small $k$ = flexible/noisy; large $k$ = smooth/biased.
- The **distance metric** and **feature scaling** strongly affect results — always standardize.
- It degrades in high dimensions (curse of dimensionality) and is slow at prediction time.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — KNN vote probabilities

A KNN classifier doesn't have to return only the winning label: the *fraction* of the k neighbors voting for class 1 is a (crude but useful) probability estimate. Implement it.

In [ ]:
def knn_proba(X_train, y_train, query, k):
    """Return P(class 1) = fraction of the k nearest neighbors with label 1."""
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train)
    query = np.asarray(query, dtype=float)

    # TODO(you): compute Euclidean distances from query to every training point
    dists = ...

    # TODO(you): indices of the k smallest distances (hint: np.argsort)
    nearest = ...

    # TODO(you): fraction of those k labels equal to 1
    return ...

In [ ]:
# Checks — run me
Xt = [[0, 0], [0, 1], [1, 0], [5, 5], [5, 6], [6, 5]]
yt = [0, 0, 0, 1, 1, 1]

assert knn_proba(Xt, yt, [0.2, 0.2], k=3) == 0.0, "all 3 nearest are class 0"
assert knn_proba(Xt, yt, [5.2, 5.2], k=3) == 1.0, "all 3 nearest are class 1"
assert knn_proba(Xt, yt, [2.5, 2.5], k=6) == 0.5, "all points used -> 3/6 vote for class 1"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def knn_proba(X_train, y_train, query, k):
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train)
    query = np.asarray(query, dtype=float)
    dists = np.linalg.norm(X_train - query, axis=1)
    nearest = np.argsort(dists)[:k]
    return float(np.mean(y_train[nearest] == 1))
```

</details>

### Exercise 2 — Distance-weighted votes

Plain KNN gives every neighbor one vote. **Weighted KNN** lets closer neighbors count for more, weighting each vote by `1 / (distance + eps)`. Implement the weighted vote for class 1.

In [ ]:
def knn_weighted_vote(X_train, y_train, query, k, eps=1e-9):
    """Return the weighted vote share for class 1 among the k nearest neighbors,
    where each neighbor's weight is 1 / (its distance + eps)."""
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train)
    query = np.asarray(query, dtype=float)

    dists = np.linalg.norm(X_train - query, axis=1)
    nearest = np.argsort(dists)[:k]

    # TODO(you): weights for the k nearest neighbors
    w = ...

    # TODO(you): (sum of weights of class-1 neighbors) / (sum of all weights)
    return ...

In [ ]:
# Checks — run me
Xt = [[0, 0], [4, 0], [5, 0]]
yt = [1, 0, 0]

# Query at (1,0): class-1 point is at distance 1; class-0 points at 3 and 4.
# Weights ≈ 1, 1/3, 1/4  ->  class-1 share = 1 / (1 + 1/3 + 1/4) ≈ 0.6316
share = knn_weighted_vote(Xt, yt, [1, 0], k=3)
assert abs(share - 12 / 19) < 1e-6, f"expected ~0.632, got {share}"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def knn_weighted_vote(X_train, y_train, query, k, eps=1e-9):
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train)
    query = np.asarray(query, dtype=float)
    dists = np.linalg.norm(X_train - query, axis=1)
    nearest = np.argsort(dists)[:k]
    w = 1.0 / (dists[nearest] + eps)
    return float(w[y_train[nearest] == 1].sum() / w.sum())
```

</details>